<a href="https://colab.research.google.com/github/sarthak-singh18/45-Days-DSA-Revision/blob/main/RR_Pressure_Performance_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from zipfile import ZipFile

In [ ]:
from zipfile import ZipFile
import os

zip_path = "/content/6af3166e7b922_data_pack.zip"
extract_path = "/content/rr_data"

with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")

print(os.listdir(extract_path))

In [ ]:
import pandas as pd
import numpy as np

# Load datasets
balls = pd.read_csv("/content/rr_data/ball_by_ball.csv")
matches = pd.read_csv("/content/rr_data/matches.csv")
competitions = pd.read_csv("/content/rr_data/competitions.csv")
players = pd.read_csv("/content/rr_data/competition_team_player_mapping.csv")
playing_xi = pd.read_csv("/content/rr_data/playing_eleven.csv")
teams = pd.read_csv("/content/rr_data/teams.csv")

print("All datasets loaded successfully!")

In [ ]:
datasets = {
    "Ball by Ball": balls,
    "Matches": matches,
    "Competitions": competitions,
    "Players": players,
    "Playing XI": playing_xi,
    "Teams": teams
}

for name, df in datasets.items():
    print(name, ":", df.shape)

In [ ]:
print("BALL-BY-BALL COLUMNS")
for col in balls.columns:
    print(col)

print("\nFIRST 5 RECORDS")
display(balls.head())

In [ ]:
print("MATCH DATA")
display(matches.head())

print("\nDATA TYPES")
print(balls.dtypes)

In [ ]:
# Check missing values
print("MISSING VALUES")
print(balls.isnull().sum())

# Check duplicate records
print("\nDUPLICATE ROWS:")
print(balls.duplicated().sum())

# Unique players
print("\nUNIQUE BATTERS:")
print(balls["striker_id"].nunique())

print("\nUNIQUE BOWLERS:")
print(balls["bowler_id"].nunique())

# Match formats
print("\nMATCH FORMATS:")
print(competitions["match_format"].value_counts())

# Seasons
print("\nSEASONS:")
print(sorted(competitions["season"].dropna().unique()))

In [ ]:
print("RUNS DISTRIBUTION")
print(balls["runs"].value_counts().sort_index())

print("\nEXTRAS TYPES")
print(balls["extras_type"].value_counts(dropna=False))

print("\nDISMISSAL TYPES")
print(balls["dismissal_type"].value_counts(dropna=False))

print("\nWICKET RECORDS")
print(balls["is_wicket"].value_counts())

In [ ]:
print("INNINGS:")
print(balls["innings"].value_counts())

print("\nOVERS:")
print(balls["overid"].describe())

print("\nREVISED TARGETS:")
print(matches["revised_target"].notna().sum())

print("\nREVISED OVERS:")
print(matches[
    ["first_inn_revised_overs", "second_inn_revised_overs"]
].notna().sum())

In [ ]:
print("EXTRAS1_RUNS:")
print(balls["extras1_runs"].value_counts(dropna=False))

print("\nEXTRAS_2_RUNS:")
print(balls["extras_2_runs"].value_counts(dropna=False))

print("\nRUNS WITH EXTRAS:")
display(
    balls.loc[
        balls["extras_type"].notna(),
        ["runs", "extras_type", "extras1_runs",
         "extras_2_runs", "is_four", "is_six"]
    ].head(20)
)

In [ ]:
extra_innings = balls[balls["innings"].isin([3, 4])]

print("Extra innings records:", len(extra_innings))

display(
    extra_innings.groupby(
        ["match_id", "innings"]
    ).size().reset_index(name="deliveries")
)

print("\nMATCH TABLE COLUMNS:")
print(matches.columns.tolist())

In [ ]:
print("COMPETITION DETAILS:")

display(
    competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ].head(20)
)

In [ ]:
print("MATCH FORMAT DISTRIBUTION")

print(
    competitions["match_format"].value_counts()
)

print("\nSEASON DISTRIBUTION")

print(
    competitions["season"].value_counts().sort_index()
)

In [ ]:
# Work with normal innings only
df = balls[balls["innings"].isin([1, 2])].copy()

# Calculate legal deliveries
df["is_legal"] = ~df["extras_type"].isin(["wide", "Overstep"])

# Calculate total wickets lost in each innings
df["wickets_before_ball"] = (
    df.groupby(["match_id", "innings"])["is_wicket"]
    .transform(lambda x: x.shift(fill_value=False).cumsum())
)

# Calculate cumulative runs
df["cumulative_runs"] = (
    df.groupby(["match_id", "innings"])["runs"]
    .cumsum()
)

# Calculate cumulative legal balls
df["legal_balls_used"] = (
    df.groupby(["match_id", "innings"])["is_legal"]
    .cumsum()
)

print(df[[
    "match_id", "innings", "overid",
    "striker_name", "bowler_name",
    "runs", "wickets_before_ball",
    "cumulative_runs", "legal_balls_used"
]].head(15))

In [ ]:
df["death_over"] = df["overid"] >= 16

df["wicket_pressure"] = (
    df["wickets_before_ball"] >= 5
)

df["basic_pressure"] = (
    df["death_over"] |
    df["wicket_pressure"]
)

print("Pressure deliveries:", df["basic_pressure"].sum())

print("Normal deliveries:", (~df["basic_pressure"]).sum())

In [ ]:
# Pressure distribution
pressure_summary = (
    df.groupby("basic_pressure")
    .agg(
        deliveries=("match_id", "count"),
        matches=("match_id", "nunique"),
        batters=("striker_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

pressure_summary["basic_pressure"] = (
    pressure_summary["basic_pressure"]
    .map({True: "Pressure", False: "Normal"})
)

display(pressure_summary)


In [ ]:
batting_analysis = (
    df.groupby(["striker_name", "basic_pressure"])
    .agg(
        runs=("runs", "sum"),
        balls=("is_legal", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_wicket", "sum")
    )
    .reset_index()
)

batting_analysis["strike_rate"] = (
    batting_analysis["runs"] /
    batting_analysis["balls"].replace(0, np.nan) * 100
)

display(batting_analysis.head(20))

In [ ]:
bowling_analysis = (
    df.groupby(["bowler_name", "basic_pressure"])
    .agg(
        runs_conceded=("runs", "sum"),
        legal_balls=("is_legal", "sum"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
)

bowling_analysis["economy"] = (
    bowling_analysis["runs_conceded"] /
    bowling_analysis["legal_balls"].replace(0, np.nan) * 6
)

display(bowling_analysis.head(20))

In [ ]:
# Aggregate batter performance
batters = (
    df.groupby(["striker_id", "striker_name", "basic_pressure"])
    .agg(
        runs=("runs", "sum"),
        balls=("is_legal", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("is_wicket", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

batters["strike_rate"] = (
    batters["runs"] / batters["balls"].replace(0, np.nan) * 100
)

batters["boundary_pct"] = (
    (batters["fours"] + batters["sixes"]) /
    batters["balls"].replace(0, np.nan) * 100
)

batters["dismissal_rate"] = (
    batters["dismissals"] /
    batters["balls"].replace(0, np.nan) * 100
)

display(batters.head(10))

In [ ]:
bowlers = (
    df.groupby(["bowler_id", "bowler_name", "basic_pressure"])
    .agg(
        runs=("runs", "sum"),
        balls=("is_legal", "sum"),
        wickets=("is_wicket", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

bowlers["economy"] = (
    bowlers["runs"] / bowlers["balls"].replace(0, np.nan) * 6
)

bowlers["wickets_per_100_balls"] = (
    bowlers["wickets"] /
    bowlers["balls"].replace(0, np.nan) * 100
)

display(bowlers.head(10))

In [ ]:
pressure_batters = batters[
    (batters["basic_pressure"] == True) &
    (batters["balls"] >= 60) &
    (batters["matches"] >= 5)
].copy()

pressure_bowlers = bowlers[
    (bowlers["basic_pressure"] == True) &
    (bowlers["balls"] >= 30) &
    (bowlers["matches"] >= 5)
].copy()

print("Eligible batters:", len(pressure_batters))
print("Eligible bowlers:", len(pressure_bowlers))

In [ ]:
# Batter comparison
batter_comparison = batters.pivot(
    index=["striker_id", "striker_name"],
    columns="basic_pressure",
    values="strike_rate"
)

batter_comparison = batter_comparison.rename(
    columns={False: "normal_sr", True: "pressure_sr"}
)

batter_comparison["sr_improvement"] = (
    batter_comparison["pressure_sr"] -
    batter_comparison["normal_sr"]
)

display(
    batter_comparison.sort_values(
        "sr_improvement", ascending=False
    ).head(15)
)

In [ ]:
# Create pressure and normal batting tables

batter_wide = batters.pivot(
    index=["striker_id", "striker_name"],
    columns="basic_pressure",
    values=["runs", "balls", "fours", "sixes",
            "dismissals", "matches", "strike_rate",
            "boundary_pct", "dismissal_rate"]
)

batter_wide.columns = [
    f"{metric}_{'pressure' if pressure else 'normal'}"
    for metric, pressure in batter_wide.columns
]

batter_wide = batter_wide.reset_index()

# Calculate improvement
batter_wide["sr_change"] = (
    batter_wide["strike_rate_pressure"] -
    batter_wide["strike_rate_normal"]
)

batter_wide["boundary_change"] = (
    batter_wide["boundary_pct_pressure"] -
    batter_wide["boundary_pct_normal"]
)

batter_wide["dismissal_change"] = (
    batter_wide["dismissal_rate_normal"] -
    batter_wide["dismissal_rate_pressure"]
)

display(batter_wide.head(10))

In [ ]:
bowler_wide = bowlers.pivot(
    index=["bowler_id", "bowler_name"],
    columns="basic_pressure",
    values=["runs", "balls", "wickets", "matches", "economy"]
)

bowler_wide.columns = [
    f"{metric}_{'pressure' if pressure else 'normal'}"
    for metric, pressure in bowler_wide.columns
]

bowler_wide = bowler_wide.reset_index()

bowler_wide["economy_improvement"] = (
    bowler_wide["economy_normal"] -
    bowler_wide["economy_pressure"]
)

display(bowler_wide.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.scatterplot(
    data=batter_wide,
    x="strike_rate_normal",
    y="strike_rate_pressure"
)

plt.title("Batter Performance: Normal vs Pressure")
plt.xlabel("Normal Strike Rate")
plt.ylabel("Pressure Strike Rate")
plt.axline((0, 0), slope=1, linestyle="--")
plt.show()

In [ ]:
sns.scatterplot(
    data=bowler_wide,
    x="economy_normal",
    y="economy_pressure"
)

plt.title("Bowler Economy: Normal vs Pressure")
plt.xlabel("Normal Economy")
plt.ylabel("Pressure Economy")
plt.axline((0, 0), slope=1, linestyle="--")
plt.show()

In [ ]:
print("ELIGIBLE BATTERS:", len(pressure_batters))
print("ELIGIBLE BOWLERS:", len(pressure_bowlers))

print("\nTOP 15 BATTERS")
display(
    batter_wide.sort_values(
        "sr_change", ascending=False
    )[[
        "striker_name",
        "balls_pressure",
        "strike_rate_normal",
        "strike_rate_pressure",
        "sr_change"
    ]].head(15)
)

print("\nTOP 15 BOWLERS")
display(
    bowler_wide.sort_values(
        "economy_improvement", ascending=False
    )[[
        "bowler_name",
        "balls_pressure",
        "economy_normal",
        "economy_pressure",
        "economy_improvement"
    ]].head(15)
)

In [ ]:
# BATTER SHORTLIST
top_batters = batter_wide[
    (batter_wide["balls_pressure"] >= 60) &
    (batter_wide["matches_pressure"] >= 5)
].sort_values(
    "sr_change", ascending=False
).head(20)

print("TOP 20 BATTER CANDIDATES")

display(
    top_batters[[
        "striker_name",
        "balls_pressure",
        "matches_pressure",
        "strike_rate_normal",
        "strike_rate_pressure",
        "sr_change",
        "boundary_change",
        "dismissal_change"
    ]]
)

# BOWLER SHORTLIST
top_bowlers = bowler_wide[
    (bowler_wide["balls_pressure"] >= 30) &
    (bowler_wide["matches_pressure"] >= 5)
].sort_values(
    "economy_improvement", ascending=False
).head(20)

print("TOP 20 BOWLER CANDIDATES")

display(
    top_bowlers[[
        "bowler_name",
        "balls_pressure",
        "matches_pressure",
        "economy_normal",
        "economy_pressure",
        "economy_improvement"
    ]]
)

In [ ]:
top_batters.to_csv(
    "/content/top_20_batter_candidates.csv",
    index=False
)

top_bowlers.to_csv(
    "/content/top_20_bowler_candidates.csv",
    index=False
)

print("Shortlists exported successfully!")

In [ ]:
from google.colab import files

files.download("/content/top_20_batter_candidates.csv")
files.download("/content/top_20_bowler_candidates.csv")

In [ ]:
from google.colab import files

files.download("/content/rr_data/ball_by_ball.csv")
files.download("/content/rr_data/matches.csv")

In [ ]:
# Decode extras categories
extras_mapping = {
    1: "Overstep",
    2: "Wide",
    3: "Legbyes",
    4: "Byes",
    5: "Other",
    6: "None",
    7: "Penalty"
}

balls["extras_category"] = (
    balls["extras1_runs"].map(extras_mapping)
)

# Identify delivery types
balls["is_wide"] = (
    balls["extras_category"] == "Wide"
)

balls["is_noball"] = (
    balls["extras_category"] == "Overstep"
)

# Batter balls faced
balls["batter_ball_faced"] = ~balls["is_wide"]

# Bowler legal deliveries
balls["bowler_legal_ball"] = ~(
    balls["is_wide"] | balls["is_noball"]
)

# Bowler credited wickets
bowler_wicket_types = [
    "Caught",
    "Bowled",
    "LBW",
    "Stump",
    "Hit Wicket"
]

balls["bowler_wicket"] = (
    balls["dismissal_type"].isin(bowler_wicket_types)
)

print("Batter balls faced:",
      balls["batter_ball_faced"].sum())

print("Bowler legal deliveries:",
      balls["bowler_legal_ball"].sum())

print("Bowler credited wickets:",
      balls["bowler_wicket"].sum())

print("\nDelivery classification:")
display(
    balls["extras_category"].value_counts()
)

In [ ]:
print("WIDE DELIVERIES:", balls["is_wide"].sum())
print("NO-BALL DELIVERIES:", balls["is_noball"].sum())

print("BATTER BALLS FACED:",
      balls["batter_ball_faced"].sum())

print("BOWLER LEGAL BALLS:",
      balls["bowler_legal_ball"].sum())

print("BOWLER-CREDITED WICKETS:",
      balls["bowler_wicket"].sum())

In [ ]:
# Regular innings only
df = balls[balls["innings"].isin([1, 2])].copy()

df["is_legal"] = df["bowler_legal_ball"]
df["batter_ball"] = df["batter_ball_faced"]

# Exclude super overs and other additional innings
print("Regular innings records:", len(df))

In [ ]:
batting_stats = (
    df.groupby(["striker_id", "striker_name"])
    .agg(
        runs=("runs", "sum"),
        balls=("batter_ball", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        matches=("match_id", "nunique"),
        dismissals=("dismissed_player_id", "count")
    )
    .reset_index()
)

batting_stats["strike_rate"] = (
    batting_stats["runs"] /
    batting_stats["balls"].replace(0, np.nan) * 100
)

batting_stats["boundary_percentage"] = (
    (batting_stats["fours"] + batting_stats["sixes"]) /
    batting_stats["balls"].replace(0, np.nan) * 100
)

display(
    batting_stats.sort_values(
        "runs", ascending=False
    ).head(20)
)

In [ ]:
bowling_stats = (
    df.groupby(["bowler_id", "bowler_name"])
    .agg(
        legal_balls=("bowler_legal_ball", "sum"),
        bowler_wickets=("bowler_wicket", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

bowling_stats["wickets_per_100_balls"] = (
    bowling_stats["bowler_wickets"] /
    bowling_stats["legal_balls"].replace(0, np.nan) * 100
)

display(
    bowling_stats.sort_values(
        "bowler_wickets", ascending=False
    ).head(20)
)

In [ ]:

# Start with regular innings
df = balls[balls["innings"].isin([1, 2])].copy()

# Recreate delivery classifications
df["is_wide"] = df["extras_type"].eq("wide")
df["is_noball"] = df["extras_type"].eq("Overstep")

# Batter balls faced: wides excluded, no-balls included
df["batter_ball"] = ~df["is_wide"]

# Wickets lost before the current delivery
df["wickets_before_ball"] = (
    df.groupby(["match_id", "innings"])["is_wicket"]
      .transform(lambda x: x.shift(fill_value=False).cumsum())
)

# Preliminary pressure classification
df["pressure"] = (
    (df["overid"] >= 16) |
    (df["wickets_before_ball"] >= 5)
)

df["pressure_label"] = df["pressure"].map({
    True: "Pressure",
    False: "Normal"
})

print(df["pressure_label"].value_counts())


In [ ]:

batting_pressure = (
    df.groupby(["striker_id", "striker_name", "pressure_label"])
      .agg(
          runs=("runs", "sum"),
          balls=("batter_ball", "sum"),
          fours=("is_four", "sum"),
          sixes=("is_six", "sum"),
          matches=("match_id", "nunique")
      )
      .reset_index()
)

batting_pressure["strike_rate"] = (
    batting_pressure["runs"] /
    batting_pressure["balls"].replace(0, np.nan) * 100
)

batting_pressure["boundary_pct"] = (
    (batting_pressure["fours"] + batting_pressure["sixes"]) /
    batting_pressure["balls"].replace(0, np.nan) * 100
)

display(
    batting_pressure.sort_values(
        ["pressure_label", "strike_rate"],
        ascending=[True, False]
    ).head(30)
)


In [ ]:

batter_comparison = batting_pressure.pivot(
    index=["striker_id", "striker_name"],
    columns="pressure_label",
    values=["runs", "balls", "matches", "strike_rate", "boundary_pct"]
)

batter_comparison.columns = [
    f"{metric}_{situation.lower()}"
    for metric, situation in batter_comparison.columns
]

batter_comparison = batter_comparison.reset_index()

batter_comparison["sr_improvement"] = (
    batter_comparison["strike_rate_pressure"] -
    batter_comparison["strike_rate_normal"]
)

display(
    batter_comparison.sort_values(
        "sr_improvement", ascending=False
    ).head(20)
)
